In [0]:
from kagglehub import dataset_download
import os
import shutil

from pyspark.sql.functions import lit, current_timestamp

In [0]:
dbutils.widgets.text("catalog", "olist_project_dev")
dbutils.widgets.text("bronze_schema", "olist_bronze")
dbutils.widgets.text("metadata_table", "olist_source_metadata")

In [0]:
catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
metadata_table_name = dbutils.widgets.get("metadata_table")

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_schema}")

In [0]:
if not spark.catalog.tableExists(f"{catalog}.{bronze_schema}.{metadata_table_name}"):
    spark.sql(
        f"""
        CREATE TABLE {catalog}.{bronze_schema}.{metadata_table_name} (
            versionNum INT,
            createdTimestamp TIMESTAMP,
            updatedTimestamp TIMESTAMP,
            isActive BOOLEAN
        )
        TBLPROPERTIES (
            'delta.autoOptimize.optimizeWrite' = 'true',
            'delta.autoOptimize.autoCompact' = 'true'
        )
        """
    )

metadata_df = spark.table(f"{catalog}.{bronze_schema}.{metadata_table_name}")

In [0]:
download_path = dataset_download("olistbr/brazilian-ecommerce")

num_version = int(download_path.split("/")[-1])

if metadata_df.filter(f"versionNum = {num_version}").count() > 0:
    dbutils.notebook.exit("Dataset already downloaded")

In [0]:
volume_path = f"/Volumes/{catalog}/{bronze_schema}/raw_data"
dbutils.fs.mkdirs(volume_path)

for csv_file in os.listdir(download_path):
    local_file = f"{download_path}/{csv_file}"
    volume_file = f"{volume_path}/{csv_file}"
    shutil.copy2(local_file, volume_file)

In [0]:
for downloaded_file in os.listdir(volume_path):
    
    table_name = downloaded_file.replace(".csv", "").replace("_dataset", "")
    
    raw_df = spark.read.csv(f"{volume_path}/{downloaded_file}", header=True, inferSchema=True)
    
    (raw_df
     .withColumns({
         "ingestionTimestamp": lit(current_timestamp()),
         "sourceFile": lit(downloaded_file)
     })
     .write
     .format("delta")
     .mode("overwrite")
     .saveAsTable(f"{catalog}.{bronze_schema}.{table_name}")
    )

In [0]:
spark.sql(
    f"""
    INSERT INTO {catalog}.{bronze_schema}.{metadata_table_name}
    VALUES ({num_version}, current_timestamp(), NULL, true)
    """
)